In [42]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.insert(0, '..')  # nbs/notebook_cache.py
from notebook_cache import NotebookCache

cache = NotebookCache('../../cache/verne')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Verne, small, step 0: the whole model, dumb, on T1

Verne V0.5 (`../verne-v0.5`): a video transformer looks at the cameras **once**, a narrow action transformer reads its keys and values at every layer and denoises an action chunk.

The code lives in `dream_robot/policies/verne/`; this notebook only runs it and looks. Every box below is the dumbest version; later notebooks open one box each.

```-
top frame ─► 16px patches ─► VIDEO transformer (4 layers, 256 wide) ── k,v per layer ─┐
joints ────► one more token ─┘   runs once per chunk                                   │
noise [16 x 8] ─► ACTION transformer (4 layers, 128 wide) ◄────────────────────────────┘
                  10 flow steps ─► 16 joint actions
```

Same data and rollouts as dreamzero's WAMz (10 labelled T1 demos): WAMz got 18 + 16 = 34/40.

### step 0.1 — the data


In [43]:
import numpy as np
from frozendict import frozendict

from dream_robot.policies.verne.data import load_table, load_top, to_gpu

NAME = 'pick_place_cube_n100'

In [44]:
@cache
def table():
    state, action, episode = load_table(NAME)  # [N, 8], [N, 8], [N]
    return frozendict(state=state, action=action, episode=episode)

table: cached, 1 call(s) stored


In [45]:
@cache
def images():
    return load_top(NAME)  # [N, 3, 128, 128] uint8, a few minutes the first time

images: cached, 1 call(s) stored


In [46]:
data = table()
val = np.random.default_rng(0).permutation(100)[:10]  # as dreamzero / uwm2
rest = np.setdiff1d(np.arange(100), val)
split = frozendict(val=val, labelled=np.random.default_rng(1).choice(rest, 10, replace=False))
demos = to_gpu(images(), data['state'], data['action'], data['episode'], split['labelled'])
print({k: len(v) for k, v in split.items()}, tuple(demos.top.shape))

OutOfMemoryError: CUDA out of memory. Tried to allocate 1000.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 66.62 MiB is free. Process 4615 has 1.37 GiB memory in use. Including non-PyTorch memory, this process has 2.65 GiB memory in use. Process 8391 has 2.79 GiB memory in use. Of the allocated memory 2.12 GiB is allocated by PyTorch, and 22.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### step 0.2 — the model


In [ ]:
import torch

from dream_robot.policies.verne.model import Verne

net = Verne()
print(f'{sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params')

3.66M params


### step 0.3 — train (flow matching on the 16-action chunk, 20k steps, ~3 min)


In [ ]:
import dream_robot.policies.verne as verne


@cache
def train_run(steps: int):
    return verne.train.train(demos, split['labelled'], split['val'], steps)

train_run: cached, 1 call(s) stored


In [ ]:
weights, log = train_run(20000)

step      0   train 2.026   val 3.142
step   1000   train 0.158   val 0.207
step   2000   train 0.073   val 0.179
step   3000   train 0.087   val 0.269
step   4000   train 0.040   val 0.177
step   5000   train 0.054   val 0.168
step   6000   train 0.057   val 0.244
step   7000   train 0.029   val 0.197
step   8000   train 0.037   val 0.189
step   9000   train 0.033   val 0.216
step  10000   train 0.038   val 0.249
step  11000   train 0.036   val 0.228
step  12000   train 0.037   val 0.183
step  13000   train 0.026   val 0.219
step  14000   train 0.045   val 0.193
step  15000   train 0.036   val 0.206
step  16000   train 0.027   val 0.201
step  17000   train 0.019   val 0.184
step  18000   train 0.020   val 0.207
step  19000   train 0.036   val 0.241
step  20000   train 0.010   val 0.246


### step 0.4 — rollouts: T1 seeds 1000-1019, replan after 8 or all 16 actions


In [ ]:
import logging
import os
from collections import Counter

os.environ.setdefault('MUJOCO_GL', 'egl')
logging.disable(logging.WARNING)
import robosuite  # noqa: F401

logging.disable(logging.NOTSET)
logging.getLogger('robosuite_logs').setLevel(logging.ERROR)
from dream_robot.core.registry import make_env
from dream_robot.policies.verne.model import sample

env = make_env('robosuite/pick_place_cube')

In [ ]:
def act(net: Verne, obs) -> np.ndarray:
    # current top frame + joints -> [16, 8] raw actions
    frame = torch.from_numpy(obs.images['top']).permute(2, 0, 1)[None].cuda().float() / 127.5 - 1
    s = (torch.from_numpy(obs.state)[None].cuda() - demos.s_mean) / demos.s_std
    chunk = (sample(net, frame, s) * demos.a_std + demos.a_mean)[0].cpu().numpy()
    chunk[:, 7] = np.clip(chunk[:, 7], 0.0, 1.0)  # gripper commands stay in [0, 1]
    return chunk


def rollout(net: Verne, seed: int, n_exec: int) -> tuple[bool, str, int]:
    obs, plan = env.reset(seed=seed), []
    for step in range(600):
        if not plan:
            plan = list(act(net, obs)[:n_exec])
        r = env.step(plan.pop(0))
        obs = r.observation
        if r.terminated or r.truncated:
            break
    return r.success, str(r.failure_mode), step + 1

In [ ]:
@cache
def sweep(n_exec: int) -> dict[int, tuple[bool, str, int]]:
    net = Verne().cuda().eval()
    net.load_state_dict(weights)
    torch.manual_seed(n_exec)
    res = {seed: rollout(net, seed, n_exec) for seed in range(1000, 1020)}
    fails = Counter(mode for ok, mode, _ in res.values() if not ok)
    print(
        f'n_exec {n_exec:2d}: {sum(ok for ok, _, _ in res.values()):2d}/20   failures {dict(fails)}'
    )
    return res

sweep: cached, 2 call(s) stored


In [ ]:
sweeps = {n_exec: sweep(n_exec) for n_exec in [8, 16]}
print('WAMz, 10 labelled (dreamzero step 24): 18, 16')

n_exec  8:  8/20   failures {'dropped': 1, 'wrong_target': 2, 'no_grasp': 9}
n_exec 16:  9/20   failures {'no_grasp': 10, 'dropped': 1}
WAMz, 10 labelled (dreamzero step 24): 18, 16
